# 02. 합성 Scratch 데이터셋 생성

이 노트북은 threshold 학습용 정답 label을 만들지 않는다. 대신 색상 조합, 폭, 길이, 각도, 휘어짐, opacity를 랜덤으로 샘플링하여 raw/mask 예시를 만든다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "scratch_postprocess_utils.py").exists():
    matches = list(Path.cwd().glob("**/Scratch_Postprocess/scratch_postprocess_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from scratch_postprocess_utils import *

ensure_dirs()
print("project:", NOTEBOOK_DIR)

In [ ]:
manifest = generate_random_scratch_dataset(
    n_samples=120,
    size=640,
    seed=7,
    overwrite=True,
)
print("samples:", len(manifest))
display(manifest.groupby("color_pair").size().reset_index(name="count"))
display(manifest.head())

## 랜덤 생성 범위 확인

In [ ]:
display(
    manifest[[
        "width_px",
        "width_um",
        "alpha",
        "length_px",
        "length_um",
        "angle_deg",
        "bend_strength",
        "n_control_points",
        "edge_roughness",
        "texture_strength",
        "color_pull",
        "gap_count",
        "blur_radius",
        "nominal_rgb_distance",
        "nominal_luma_delta",
    ]].describe()
)

## 샘플 시각화

In [ ]:
sample_view = manifest.groupby("color_pair", group_keys=False).sample(n=4, random_state=7).reset_index(drop=True)
fig, axes = plt.subplots(len(sample_view), 2, figsize=(7, 2.2 * len(sample_view)))
for row_idx, row in sample_view.iterrows():
    img = load_image(row["image_path"])
    mask = load_mask(row["mask_path"])
    axes[row_idx, 0].imshow(img)
    axes[row_idx, 0].set_title(
        f'{row["sample_id"]} | {row["color_pair"]} | a={row["alpha"]:.2f}, pull={row["color_pull"]:.2f}, w={row["width_px"]:.1f}px'
    )
    axes[row_idx, 0].axis("off")
    axes[row_idx, 1].imshow(mask, cmap="gray")
    axes[row_idx, 1].set_title(f'mask | angle={row["angle_deg"]:.1f}, ctrl={row["n_control_points"]}')
    axes[row_idx, 1].axis("off")
plt.tight_layout()
plt.savefig(RUNS_ROOT / "synthetic_random_samples_preview.png", dpi=150)
plt.show()